In [18]:
import random
import numpy as np
import torch

seed = 42

random.seed(seed)                  # Python built-in random
np.random.seed(seed)               # NumPy
torch.manual_seed(seed)            # PyTorch (CPU)
torch.cuda.manual_seed(seed)       # PyTorch (single GPU)
torch.cuda.manual_seed_all(seed)   # PyTorch (all GPUs)

# Ensures deterministic behavior
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [19]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

class CustomDataset(Dataset):
    def __init__(self, file_paths, transform=None):
        self.file_paths = file_paths
        self.transform = transform
        self.file_names = [os.path.basename(path) for path in file_paths]

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        data = torch.load(self.file_paths[idx], weights_only=True)

        images = data[:6]
        labels = data[6]

        images = images.float()
        labels = labels.long()
        labels = labels + 1

        if self.transform:
            images = self.transform(images)
            labels = self.transform(labels)

        return images, labels, self.file_names[idx]

class CustomDataset_test(Dataset):
    def __init__(self, file_paths, transform=None):
        self.file_paths = file_paths
        self.transform = transform
        self.file_names = [os.path.basename(path) for path in file_paths]

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        data = torch.load(self.file_paths[idx], weights_only=True)

        images = data[:6]

        images = images.float()

        if self.transform:
            images = self.transform(images)

        return images, self.file_names[idx]

def generate_file_paths(base_path):
    file_paths = []
    for frame in os.listdir(base_path):
        frame_path = os.path.join(base_path, frame)
        if frame_path.endswith('.mat.pt'):
            file_paths.append(frame_path)
    return [path for path in file_paths if os.path.exists(path)]

def load_data(base_path, batch_size=4, num_workers=2, test_size=0.2):
    file_paths = generate_file_paths(base_path)

    train_paths, test_paths = train_test_split(file_paths, test_size=test_size, random_state=42)

    train_dataset = CustomDataset(file_paths=train_paths)
    test_dataset = CustomDataset(file_paths=test_paths)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        drop_last=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        drop_last=True
    )

    return train_loader, test_loader

In [20]:
import torch
import torch.nn as nn
import torch.optim as optim

class MyModel(nn.Module):
    def __init__(self):
        super(MyModel, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.conv5 = nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, padding=1)
        self.conv6 = nn.Conv2d(in_channels=256, out_channels=128, kernel_size=3, padding=1)
        self.conv7 = nn.Conv2d(in_channels=128, out_channels=64, kernel_size=3, padding=1)
        self.conv8 = nn.Conv2d(in_channels=64, out_channels=32, kernel_size=3, padding=1)
        self.conv9 = nn.Conv2d(in_channels=32, out_channels=5, kernel_size=3, padding=1)

        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.relu(self.conv3(x))
        x = self.relu(self.conv4(x))
        x = self.relu(self.conv5(x))
        x = self.relu(self.conv6(x))
        x = self.relu(self.conv7(x))
        x = self.relu(self.conv8(x))
        x = self.conv9(x)
        return x

def train(model, train_loader, test_loader, optimizer, criterion, num_epochs=100):
    train_losses = []
    val_losses = []

    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0
        batch_count = 0

        for images, labels, _ in train_loader:
            images = images.cuda() if torch.cuda.is_available() else images
            labels = labels.cuda() if torch.cuda.is_available() else labels

            outputs = model(images)
            outputs = outputs.view(outputs.size(0), outputs.size(1), -1)  # [B, C, H*W]
            labels = labels.view(labels.size(0), -1)  # [B, H*W]
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            batch_count += 1

        avg_train_loss = epoch_loss / batch_count
        train_losses.append(avg_train_loss)

        model.eval()
        val_loss = 0.0
        val_batch_count = 0

        with torch.no_grad():
            for images, labels, _ in test_loader:
                images = images.cuda() if torch.cuda.is_available() else images
                labels = labels.cuda() if torch.cuda.is_available() else labels

                outputs = model(images)
                outputs = outputs.view(outputs.size(0), outputs.size(1), -1)
                labels = labels.view(labels.size(0), -1)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                val_batch_count += 1

        avg_val_loss = val_loss / val_batch_count
        val_losses.append(avg_val_loss)

        if (epoch+1) % 2 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], '
                  f'Train Loss: {avg_train_loss:.4f}, '
                  f'Val Loss: {avg_val_loss:.4f}')

    return train_losses, val_losses
TRAIN_PATH = "./"
# The training set is deployed automatically in the testing machine.
# Your notebook can access the TRAIN_PATH even if you do not mount it along with notebook.
data_path = TRAIN_PATH + 'training_set'

train_loader, test_loader = load_data(
    base_path=data_path,
    batch_size=4,
    num_workers=2,
    test_size=0.2
)

model = MyModel()
if torch.cuda.is_available():
    model = model.cuda()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

train_losses, val_losses = train(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    optimizer=optimizer,
    criterion=criterion,
    num_epochs=40
)

Epoch [2/40], Train Loss: 0.0687, Val Loss: 0.0563
Epoch [4/40], Train Loss: 0.0460, Val Loss: 0.0452
Epoch [6/40], Train Loss: 0.0390, Val Loss: 0.0396
Epoch [8/40], Train Loss: 0.0346, Val Loss: 0.0345
Epoch [10/40], Train Loss: 0.0317, Val Loss: 0.0329
Epoch [12/40], Train Loss: 0.0299, Val Loss: 0.0304
Epoch [14/40], Train Loss: 0.0277, Val Loss: 0.0293
Epoch [16/40], Train Loss: 0.0263, Val Loss: 0.0289
Epoch [18/40], Train Loss: 0.0250, Val Loss: 0.0289
Epoch [20/40], Train Loss: 0.0237, Val Loss: 0.0273
Epoch [22/40], Train Loss: 0.0226, Val Loss: 0.0272
Epoch [24/40], Train Loss: 0.0216, Val Loss: 0.0276
Epoch [26/40], Train Loss: 0.0209, Val Loss: 0.0300
Epoch [28/40], Train Loss: 0.0200, Val Loss: 0.0267
Epoch [30/40], Train Loss: 0.0193, Val Loss: 0.0264
Epoch [32/40], Train Loss: 0.0184, Val Loss: 0.0267
Epoch [34/40], Train Loss: 0.0178, Val Loss: 0.0269
Epoch [36/40], Train Loss: 0.0172, Val Loss: 0.0278
Epoch [38/40], Train Loss: 0.0166, Val Loss: 0.0282
Epoch [40/40], T

In [21]:
import torch
import torch.nn.functional as F

def radar_outputs(model, images):
    """DIMSSNet: six [B, 1, 50, 181] inputs, five output channels."""
    if images.ndim != 4 or images.shape[1] != 6:
        raise ValueError("Expected images [B, 6, H, W]")
    outputs = model(images)
    if outputs.ndim != 4 or outputs.shape[:2] != (images.shape[0], 5):
        raise ValueError("Expected model output [B, 5, H, W]")
    if not torch.isfinite(outputs).all():
        raise ValueError("Model output contains NaN/Inf")
    return F.interpolate(outputs, size=images.shape[-2:],
                         mode="bilinear", align_corners=False)


def evaluate(model, loader):
    """Official weighted accuracy on a labeled loader; labels are shifted to 0..4."""
    if getattr(loader, "drop_last", False):
        raise ValueError("Evaluation requires drop_last=False")
    device = next(model.parameters()).device
    was_training = model.training
    correct_score = 0
    maximum_score = 0
    model.eval()
    try:
        with torch.inference_mode():
            for images, labels, _ in loader:
                images = images.to(device=device, dtype=torch.float32)
                labels = labels.to(device)
                if labels.shape != (images.shape[0], *images.shape[-2:]):
                    raise ValueError("Expected labels [B, H, W]")
                if labels.is_floating_point() or torch.any((labels < 0) | (labels > 4)):
                    raise ValueError("Expected integer labels 0..4 (background=0)")
                predictions = radar_outputs(model, images).argmax(dim=1)
                weights = torch.where(labels == 0, 1, 50)
                correct_score += ((predictions == labels) * weights).sum().item()
                maximum_score += weights.sum().item()
    finally:
        model.train(was_training)
    if maximum_score == 0:
        raise ValueError("Cannot evaluate an empty loader")
    return correct_score / maximum_score


# test_loader here is the holdout from training_set, not the official leaderboard.
# Rebuild it to include the final incomplete batch, even in an existing kernel.
holdout_loader = DataLoader(test_loader.dataset, batch_size=4, shuffle=False,
                            num_workers=0, drop_last=False)
score = evaluate(model, holdout_loader)
print(f"Training holdout weighted score: {score:.6f}")

Training holdout weighted score: 0.844981


In [22]:
# Official leaderboard inference: inputs only, no ground-truth access.
from pathlib import Path
import os
import pandas as pd

def run_inference(model, data_loader):
    if getattr(data_loader, "drop_last", False):
        raise ValueError("Inference requires drop_last=False")
    device = next(model.parameters()).device
    was_training = model.training
    predictions, filenames = [], []
    model.eval()
    try:
        with torch.inference_mode():
            for images, file_names in data_loader:
                images = images.to(device=device, dtype=torch.float32)
                preds = radar_outputs(model, images).argmax(dim=1) - 1
                predictions.extend(preds.cpu().numpy().reshape(len(preds), -1))
                filenames.extend(file_names)
    finally:
        model.train(was_training)
    return predictions, filenames


# Locate the local copy independently of the kernel working directory.
radar_dir = next((root / "training/ioai/2025/Radar"
                  for root in [Path.cwd(), *Path.cwd().parents]
                  if (root / "training/ioai/2025/Radar/Solution").is_dir()), None)
if radar_dir is None and (Path.cwd() / "Solution").is_dir():
    radar_dir = Path.cwd()
if os.environ.get("DATA_PATH"):
    inference_root = Path(os.environ["DATA_PATH"])
elif radar_dir is not None:
    inference_root = radar_dir / "Solution"
else:
    raise FileNotFoundError("Set DATA_PATH to the folder with validation_set and test_set")

pixel_columns = [f"pixel_{i}" for i in range(50 * 181)]
for split, csv_name in [("validation_set", "submission_val.csv"),
                        ("test_set", "submission_test.csv")]:
    split_path = inference_root / split
    # Some evaluation environments call the test directory testing_set.
    if split == "test_set" and not split_path.is_dir():
        split_path = inference_root / "testing_set"
    paths = sorted(split_path.glob("*.mat.pt"), key=lambda p: int(p.name.split(".")[0]))
    if len(paths) != 500:
        raise ValueError(f"{split_path}: expected 500 samples, got {len(paths)}")
    loader = DataLoader(CustomDataset_test([str(p) for p in paths]), batch_size=4,
                        shuffle=False, num_workers=0, drop_last=False)
    predictions, filenames = run_inference(model, loader)
    frame = pd.DataFrame(predictions, columns=pixel_columns)
    frame.insert(0, "filename", filenames)
    frame.to_csv(csv_name, index=False)
    print(f"{csv_name}: {frame.shape}")

submission_val.csv: (500, 9051)
submission_test.csv: (500, 9051)


In [23]:
# Run inference on validation set and testing set
from torch.utils.data import DataLoader
import pandas as pd

def run_inference(model, data_loader):
    """Run inference and return predictions with filenames"""
    model.eval()
    predictions = []
    filenames = []

    with torch.no_grad():
        for images, file_names in data_loader:
            images = images.cuda() if torch.cuda.is_available() else images

            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)

            # Convert predictions back to original label range [-1, 3]
            preds = preds - 1

            # Flatten predictions for each sample
            for i, pred in enumerate(preds):
                predictions.append(pred.cpu().numpy().flatten())
                filenames.append(file_names[i])

    return predictions, filenames

#DATA_PATH is the secret environment variable to point the address of the validation set and test set on the testing machine.
#You cannot access this address locally.
if os.environ.get('DATA_PATH'):
    DATA_PATH = os.environ.get("DATA_PATH") + "/"
else:
    DATA_PATH = "Solution/"  # Fallback for local testing
# Load validation set
val_paths = generate_file_paths(DATA_PATH + 'validation_set')
val_dataset = CustomDataset_test(file_paths=val_paths)
val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=2
)

# Load testing set
test_paths = generate_file_paths(DATA_PATH + 'test_set')
test_dataset = CustomDataset_test(file_paths=test_paths)
test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=2
)

# Run inference on validation set
print("Running inference on validation set...")
val_predictions, val_filenames = run_inference(model, val_loader)

# Save validation results to CSV
val_results = []
for filename, pred in zip(val_filenames, val_predictions):
    # Create a row with filename and flattened predictions
    row = {'filename': filename}
    for i, p in enumerate(pred):
        row[f'pixel_{i}'] = p
    val_results.append(row)

val_df = pd.DataFrame(val_results)
val_df.to_csv('submission_val.csv', index=False)
print(f"Validation results saved to output_validation.csv with shape: {val_df.shape}")

# Run inference on testing set
print("Running inference on testing set...")
test_predictions, test_filenames = run_inference(model, test_loader)

# Save testing results to CSV
test_results = []
for filename, pred in zip(test_filenames, test_predictions):
    # Create a row with filename and flattened predictions
    row = {'filename': filename}
    for i, p in enumerate(pred):
        row[f'pixel_{i}'] = p
    test_results.append(row)

test_df = pd.DataFrame(test_results)
test_df.to_csv('submission_test.csv', index=False)
print(f"Testing results saved to output_testing.csv with shape: {test_df.shape}")

print("\nInference completed! Results saved to:")
print("- submission_val.csv (for validation set leaderboard)")
print("- submission_test.csv (for testing set leaderboard)")

Running inference on validation set...
Validation results saved to output_validation.csv with shape: (500, 9051)
Running inference on testing set...
Testing results saved to output_testing.csv with shape: (500, 9051)

Inference completed! Results saved to:
- submission_val.csv (for validation set leaderboard)
- submission_test.csv (for testing set leaderboard)


In [24]:
import zipfile
import os

# Define the files to zip and the zip file name.
files_to_zip = ['submission_val.csv', 'submission_test.csv']
zip_filename = 'submission.zip'

# Create a zip file
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in files_to_zip:
        # Add the file to the zip fil
        zipf.write(file, os.path.basename(file))

print(f'{zip_filename} Created successfully!')

submission.zip Created successfully!


In [25]:
# Run after generating submission_val.csv and submission_test.csv.
# This local-only cell is not needed on the competition evaluation server.
import importlib.util
from pathlib import Path

if radar_dir is None:
    raise FileNotFoundError("Local replay requires the published Solution/Scoring files")
spec = importlib.util.spec_from_file_location("radar_leaderboard", radar_dir / "leaderboard.py")
leaderboard = importlib.util.module_from_spec(spec)
spec.loader.exec_module(leaderboard)
leaderboard_scores = leaderboard.score_submission(Path.cwd())
print(leaderboard_scores)  # public_a and private_b, each in [0, 1]

Loaded 500 samples from CSV
Loaded 500 samples from CSV
{'public_a': 0.8471800108684007, 'private_b': 0.8503873688521534}
